In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
import warnings
import re
import unicodedata
import json
import time
warnings.filterwarnings('ignore')
print('Imports ready')

Imports ready


In [2]:
DATA_DIR = Path.home() / 'Downloads'
FILES = [
    'HIAG_unified_corrected_labels.xlsx',
    'HHM_unified_corrected_labels.xlsx',
    'ELP_unified_corrected_labels (1).xlsx',
    'Burckhardt_unified_corrected_labels.xlsx',
    'BB07_unified_corrected_labels.xlsx',
    'Spiez_BIM_data_export_unified_corrected_labels.xlsx',
]
COLUMN_MAP = {
    'globalid': ['globalid', 'global_id', 'guid'],
    'element_type': ['elementtype', 'element_type', 'element type', 'ifctype', 'ifc_type'],
    'object_type': ['objecttype', 'object_type', 'object type'],
    'name': ['name', 'element_name', 'elementname', 'object_name'],
    'description': ['description', 'desc', 'bezeichnung'],
    'signature': ['signature'],
    'label_code': ['label code', 'labelcode', 'label_code'],
    'label_name': ['label name', 'labelname', 'label_name'],
    'validation': ['validation status', 'validationstatus', 'validation_status'],
    'volume': ['volume'],
    'area': ['surfacearea', 'surface_area', 'area'],
    'length': ['length'],
    'width': ['width'],
    'height': ['height'],
    'objectid': ['objectid', 'object_id'],
}
def normalize_columns(df):
    rename_dict = {}
    df_cols_lower = {col.lower().strip(): col for col in df.columns}
    for canonical, variants in COLUMN_MAP.items():
        for variant in variants:
            if variant in df_cols_lower:
                original_col = df_cols_lower[variant]
                if original_col != canonical:
                    rename_dict[original_col] = canonical
                break
    return df.rename(columns=rename_dict)
all_dfs = []
for fname in FILES:
    fpath = DATA_DIR / fname
    if not fpath.exists():
        print(f'Skipping missing file: {fname}')
        continue
    print(f'Loading {fname}...')
    try:
        df = pd.read_excel(fpath, sheet_name='All Objects', engine='openpyxl')
    except Exception:
        df = pd.read_excel(fpath, sheet_name=0, engine='openpyxl')
    df = normalize_columns(df)
    df['source_file'] = fname
    all_dfs.append(df)
    print(f'  -> {len(df):,} rows')
master_df = pd.concat(all_dfs, ignore_index=True)
valid_df = master_df[
    master_df['validation'].astype(str).str.lower().str.strip().isin(['passed', 'corrected']) &
    master_df['label_code'].notna() &
    master_df['signature'].notna()
].copy()
print(f'Master rows: {len(master_df):,}')
print(f'Valid rows: {len(valid_df):,}')


Loading HIAG_unified_corrected_labels.xlsx...
  -> 10,936 rows
Loading HHM_unified_corrected_labels.xlsx...
  -> 63,279 rows
Loading ELP_unified_corrected_labels (1).xlsx...
  -> 27,035 rows
Loading Burckhardt_unified_corrected_labels.xlsx...
  -> 63,450 rows
Loading BB07_unified_corrected_labels.xlsx...
  -> 190,113 rows
Loading Spiez_BIM_data_export_unified_corrected_labels.xlsx...
  -> 40,064 rows
Master rows: 394,877
Valid rows: 389,795


In [3]:
def normalize_text_value(x):
    if pd.isna(x):
        return ''
    x = unicodedata.normalize('NFKC', str(x)).lower()
    x = re.sub(r'[^a-z0-9\s\./]', ' ', x)
    return re.sub(r'\s+', ' ', x).strip()
def top_unique_join(series, k=20):
    vals = [v for v in pd.Series(series.dropna().astype(str)).unique() if v]
    return ' '.join(vals[:k])
def aggregate_signatures(df):
    work = df.copy()
    num_cols = [c for c in ['volume', 'area', 'length', 'width', 'height'] if c in work.columns]
    for col in num_cols:
        work[col] = pd.to_numeric(work[col], errors='coerce').clip(lower=0)
    for col in ['name', 'description', 'object_type', 'element_type']:
        if col in work.columns:
            work[col] = work[col].map(normalize_text_value)
    grouped = work.groupby('signature')
    out = grouped.agg(
        label_code=('label_code', lambda x: x.mode().iloc[0]),
        label_name=('label_name', lambda x: x.mode().iloc[0] if not x.mode().empty else ''),
        raw_object_count=('objectid', 'count'),
        element_type=('element_type', lambda x: x.mode().iloc[0] if not x.mode().empty else 'missing'),
        object_type=('object_type', lambda x: x.mode().iloc[0] if not x.mode().empty else 'missing'),
        name=('name', lambda x: top_unique_join(x, 15)),
        description=('description', lambda x: top_unique_join(x, 20))
    ).reset_index()
    for col in num_cols:
        g = grouped[col].agg(['mean', 'max', 'min', 'std', 'sum'])
        g.columns = [f'{col}_{s}' for s in g.columns]
        if f'{col}_sum' in g.columns:
            g[f'{col}_sum_log1p'] = np.log1p(g[f'{col}_sum'].clip(lower=0))
        out = out.merge(g.reset_index(), on='signature', how='left')
    out['object_count'] = out['raw_object_count'].clip(upper=500)
    out['objectcount_log1p'] = np.log1p(out['raw_object_count'])
    out['is_huge_signature'] = (out['raw_object_count'] >= 100).astype(int)
    out['text_blob'] = (
        out['name'].fillna('') + ' [SEP] ' +
        out['description'].fillna('') + ' [SEP] ' +
        out['object_type'].fillna('') + ' [SEP] ' +
        out['element_type'].fillna('') + ' [SEP] objects ' +
        out['raw_object_count'].astype(str)
    ).str.replace(r'\s+', ' ', regex=True).str.strip()
    return out
sig_df = aggregate_signatures(valid_df)
counts = sig_df['label_code'].value_counts()
sig_df = sig_df[sig_df['label_code'].isin(counts[counts >= 2].index)].copy()
print(sig_df.shape)
print(sig_df[['raw_object_count', 'object_count', 'is_huge_signature']].describe())


(3119, 42)
       raw_object_count  object_count  is_huge_signature
count       3119.000000   3119.000000        3119.000000
mean         124.462007     32.522283           0.074383
std         1222.864057     96.477828           0.262435
min            1.000000      1.000000           0.000000
25%            1.000000      1.000000           0.000000
50%            3.000000      3.000000           0.000000
75%           10.000000     10.000000           0.000000
max        43295.000000    500.000000           1.000000


In [4]:
from sklearn.model_selection import train_test_split
y = sig_df['label_code']
cat_cols = [c for c in ['element_type', 'object_type'] if c in sig_df.columns]
num_cols = [
    c for c in sig_df.columns
    if any(c.startswith(p) for p in ['volume', 'area', 'length', 'width', 'height'])
    or c in ['object_count', 'raw_object_count', 'objectcount_log1p', 'is_huge_signature', 'footprint_ratio', 'height_length_ratio']
    or c.startswith('log1p_')
    or c.endswith('missing_ratio')
    or c.endswith('nonzero_ratio')
]
num_cols = [c for c in num_cols if c in sig_df.columns]
X = sig_df[['text_blob'] + cat_cols + num_cols].copy()
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
print(f'Train: {len(X_train):,} | Test: {len(X_test):,} | Classes: {y.nunique()}')


Train: 2,495 | Test: 624 | Classes: 97


In [5]:
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.preprocessing import OneHotEncoder, RobustScaler, LabelEncoder
from sklearn.impute import SimpleImputer
from lightgbm import LGBMClassifier
from sklearn.metrics import accuracy_score, f1_score, classification_report
EXPERIMENT_NAME = 'exp3_huge_signature_mitigation'
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(y_train)
y_test_enc = label_encoder.transform(y_test)
text_union = FeatureUnion([
    ('word', TfidfVectorizer(max_features=10000, ngram_range=(1, 2), sublinear_tf=True, min_df=2)),
    ('char', TfidfVectorizer(max_features=3000, analyzer='char_wb', ngram_range=(4, 5), sublinear_tf=True, min_df=2)),
])
preprocessor = ColumnTransformer([
    ('text', text_union, 'text_blob'),
    ('cat', Pipeline([
        ('imp', SimpleImputer(strategy='constant', fill_value='missing')),
        ('ohe', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ]), cat_cols),
    ('num', Pipeline([
        ('imp', SimpleImputer(strategy='median')),
        ('scl', RobustScaler())
    ]), num_cols),
])
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)
model = LGBMClassifier(
    n_estimators=800,
    learning_rate=0.04,
    num_leaves=63,
    min_child_samples=8,
    reg_alpha=0.2,
    reg_lambda=0.2,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1,
    verbose=-1
)
t0 = time.time()
model.fit(X_train_t, y_train_enc)
t1 = time.time()
preds = model.predict(X_test_t)
probas = model.predict_proba(X_test_t)
top3_idx = np.argsort(probas, axis=1)[:, -3:][:, ::-1]
pred_labels = label_encoder.inverse_transform(preds)
results_df = pd.DataFrame({
    'signature': X_test.index,
    'predicted_label': pred_labels,
    'confidence': probas[np.arange(len(preds)), preds],
    'alternative_1': label_encoder.inverse_transform(top3_idx[:, 1]),
    'alt_1_confidence': probas[np.arange(len(preds)), top3_idx[:, 1]],
    'alternative_2': label_encoder.inverse_transform(top3_idx[:, 2]),
    'alt_2_confidence': probas[np.arange(len(preds)), top3_idx[:, 2]],
    'true_label': y_test.values
})
results_df['correct'] = results_df['predicted_label'] == results_df['true_label']
metrics = {
    'experiment': EXPERIMENT_NAME,
    'accuracy': round(accuracy_score(y_test_enc, preds), 4),
    'macro_f1': round(f1_score(y_test_enc, preds, average='macro'), 4),
    'weighted_f1': round(f1_score(y_test_enc, preds, average='weighted'), 4),
    'mean_confidence': round(float(results_df['confidence'].mean()), 4),
    'train_time_s': round(t1 - t0, 1),
    'train_signatures': int(len(X_train)),
    'test_signatures': int(len(X_test)),
    'num_classes': int(y.nunique())
}
print(metrics)
print(classification_report(y_test, pred_labels, zero_division=0))


{'experiment': 'exp3_huge_signature_mitigation', 'accuracy': 0.8846, 'macro_f1': 0.6375, 'weighted_f1': 0.8894, 'mean_confidence': 0.9183, 'train_time_s': 61.3, 'train_signatures': 2495, 'test_signatures': 624, 'num_classes': 97}
                precision    recall  f1-score   support

         EL.01       0.00      0.00      0.00         1
         EL.04       1.00      1.00      1.00         4
         EL.09       0.00      0.00      0.00         1
      EL.09.01       0.00      0.00      0.00         1
         EL.10       0.00      0.00      0.00         0
      EQUIP.08       0.50      0.33      0.40         3
     F01.04.01       1.00      0.98      0.99        88
     F01.05.01       1.00      0.75      0.86         8
     F01.05.07       1.00      1.00      1.00         1
        F01.06       1.00      1.00      1.00         4
        F01.07       0.00      0.00      0.00         1
        F03.02       0.93      0.93      0.93        15
     F03.02.01       0.00      0.00      

In [6]:
from sklearn.metrics import classification_report

OUTPUT_DIR = Path.cwd() / 'bim_outputs' / 'final'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

results_df.to_csv(OUTPUT_DIR / 'predictions.csv', index=False)
pd.DataFrame([metrics]).to_csv(OUTPUT_DIR / 'metrics.csv', index=False)

with open(OUTPUT_DIR / 'metrics.json', 'w', encoding='utf-8') as f:
    json.dump(metrics, f, indent=2)

report_dict = classification_report(
    y_test,
    pred_labels,
    zero_division=0,
    output_dict=True
)

report_df = pd.DataFrame(report_dict).transpose()
report_df.to_csv(OUTPUT_DIR / 'classification_report.csv', index=True)

print(f'Predictions generated: {len(results_df):,}')
print(f'Overall accuracy: {metrics["accuracy"]:.4f}')


Predictions generated: 624
Overall accuracy: 0.8846


In [7]:
from pathlib import Path
import json
import pandas as pd

OUT_DIR = Path("bim_outputs/final")
OUT_DIR.mkdir(parents=True, exist_ok=True)

model_summary = {
    "experiment": metrics.get("experiment"),
    "model_name": "LightGBM",
    "n_estimators": getattr(model, "n_estimators", None),
    "learning_rate": getattr(model, "learning_rate", None),
    "num_leaves": getattr(model, "num_leaves", None),
    "min_child_samples": getattr(model, "min_child_samples", None),
    "reg_alpha": getattr(model, "reg_alpha", None),
    "reg_lambda": getattr(model, "reg_lambda", None),
    "class_weight": getattr(model, "class_weight", None),
    "random_state": getattr(model, "random_state", None),

    "accuracy": float(metrics.get("accuracy")),
    "macro_f1": float(metrics.get("macro_f1")),
    "weighted_f1": float(metrics.get("weighted_f1")),
    "mean_confidence": float(metrics.get("mean_confidence")),
    "train_time_s": float(metrics.get("train_time_s")),

    "train_signatures": int(metrics.get("train_signatures")),
    "test_signatures": int(metrics.get("test_signatures")),
    "num_classes": int(metrics.get("num_classes")),

    "feature_matrix_shape": {
        "rows": int(X_train_t.shape[0]),
        "cols": int(X_train_t.shape[1]),
    },

    "text_features": {
        "method": "FeatureUnion(TF-IDF word + char)",
        "word_tfidf": {
            "max_features": text_union.transformer_list[0][1].max_features,
            "ngram_range": list(text_union.transformer_list[0][1].ngram_range),
            "sublinear_tf": bool(text_union.transformer_list[0][1].sublinear_tf),
            "min_df": int(text_union.transformer_list[0][1].min_df),
        },
        "char_tfidf": {
            "max_features": text_union.transformer_list[1][1].max_features,
            "analyzer": text_union.transformer_list[1][1].analyzer,
            "ngram_range": list(text_union.transformer_list[1][1].ngram_range),
            "sublinear_tf": bool(text_union.transformer_list[1][1].sublinear_tf),
            "min_df": int(text_union.transformer_list[1][1].min_df),
        },
    },

    "categorical_features": {
        "encoder": "OneHotEncoder",
        "columns": list(cat_cols),
        "handle_unknown": "ignore",
    },

    "numerical_features": {
        "scaler": "RobustScaler",
        "imputation": "median",
        "columns": list(num_cols),
    },

    "split_strategy": "Signature-level stratified 80/20",
    "leakage_prevention": "No row-level splitting",
    "projects_covered": list(FILES) if "FILES" in globals() else None,
    "label_encoding": {
        "num_encoded_classes": int(len(label_encoder.classes_)),
    },
}

with open(OUT_DIR / "model_summary.json", "w") as f:
    json.dump(model_summary, f, indent=2)

print(f"wrote {OUT_DIR / 'model_summary.json'}")


wrote bim_outputs/final/model_summary.json


In [8]:
dashboard_streamlit_updated_code = '''
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
from pathlib import Path
import json
from sklearn.metrics import confusion_matrix

st.set_page_config(page_title="BIM Label Classifier Dashboard", layout="wide")

st.markdown("""
<style>
.block-container {
    padding-top: 2rem;
    padding-bottom: 3rem;
    max-width: 1400px;
}
div[data-testid="stMetric"] {
    background: rgba(255,255,255,0.02);
    border: 1px solid rgba(255,255,255,0.08);
    padding: 1rem 1.1rem;
    border-radius: 14px;
}
section.main > div {
    padding-top: 0rem;
}
</style>
""", unsafe_allow_html=True)

BASE_DIR = Path("bim_outputs/final")
PRED_PATH = BASE_DIR / "predictions.csv"
METRICS_PATH = BASE_DIR / "metrics.csv"
CLASS_METRICS_PATH = BASE_DIR / "classification_report.csv"
SUMMARY_PATH = BASE_DIR / "model_summary.json"

@st.cache_data
def load_data():
    missing = [str(p) for p in [PRED_PATH, METRICS_PATH] if not p.exists()]
    if missing:
        raise FileNotFoundError("Missing required files: " + ", ".join(missing))

    pred_df = pd.read_csv(PRED_PATH)
    metrics_df = pd.read_csv(METRICS_PATH)

    class_metrics_df = None
    if CLASS_METRICS_PATH.exists():
        class_metrics_df = pd.read_csv(CLASS_METRICS_PATH)

    summary = {}
    if SUMMARY_PATH.exists():
        with open(SUMMARY_PATH) as f:
            loaded = json.load(f)
            if isinstance(loaded, dict):
                summary = loaded

    return pred_df, metrics_df, class_metrics_df, summary

def fmt_int_or_dash(value):
    return f"{int(value):,}" if pd.notnull(value) else "—"

def fmt_float_or_dash(value, decimals=1):
    return f"{float(value):.{decimals}f}" if pd.notnull(value) else "—"

def fmt_text_or_dash(value):
    if value is None:
        return "—"
    if isinstance(value, float) and pd.isna(value):
        return "—"
    if isinstance(value, (list, tuple)) and len(value) == 0:
        return "—"
    return str(value)

try:
    pred_df, metrics_df, class_metrics_df, summary = load_data()
except Exception as e:
    st.error(f"Could not load dashboard inputs: {e}")
    st.stop()

metrics_row = metrics_df.iloc[0]

accuracy_val = summary.get("accuracy", metrics_row["accuracy"])
macro_f1_val = summary.get("macro_f1", metrics_row["macro_f1"])
weighted_f1_val = summary.get("weighted_f1", metrics_row["weighted_f1"])
test_signatures_val = summary.get("test_signatures", len(pred_df))
train_signatures_val = summary.get("train_signatures", np.nan)
num_classes_val = summary.get("num_classes", pred_df["true_label"].nunique())
mean_confidence_val = summary.get("mean_confidence", pred_df["confidence"].mean() if "confidence" in pred_df.columns else np.nan)

st.title("BIM Label Classifier — Evaluation Dashboard")
st.caption(
    "LightGBM classifier with signature-level evaluation, confidence analysis, per-class performance, "
    "confusion structure, and misclassification review."
)

c1, c2, c3, c4, c5, c6 = st.columns(6)
c1.metric("Accuracy", f"{accuracy_val*100:.2f}%")
c2.metric("Macro F1", f"{macro_f1_val:.4f}")
c3.metric("Weighted F1", f"{weighted_f1_val:.4f}")
c4.metric("Test Signatures", fmt_int_or_dash(test_signatures_val))
c5.metric("Train Signatures", fmt_int_or_dash(train_signatures_val))
c6.metric("Label Classes", fmt_int_or_dash(num_classes_val))

st.markdown("---")

st.subheader("Prediction Confidence Analysis")
col1, col2 = st.columns(2)

with col1:
    fig = px.histogram(
        pred_df,
        x="confidence",
        color="correct",
        nbins=40,
        barmode="overlay",
        color_discrete_map={True: "#2ecc71", False: "#e74c3c"},
        title="Confidence Distribution (Correct vs Incorrect)",
        labels={"confidence": "Confidence Score", "count": "Count"}
    )
    fig.update_layout(height=420, legend_title_text="Prediction result", margin=dict(t=60, b=40))
    st.plotly_chart(fig, use_container_width=True)

with col2:
    pred_df["conf_bin"] = pd.cut(pred_df["confidence"], bins=10)
    conf_acc = pred_df.groupby("conf_bin", observed=True)["correct"].mean().reset_index()
    conf_acc["conf_bin"] = conf_acc["conf_bin"].astype(str)

    fig2 = px.bar(
        conf_acc,
        x="conf_bin",
        y="correct",
        title="Accuracy by Confidence Bucket",
        labels={"conf_bin": "Confidence Range", "correct": "Accuracy"},
        color="correct",
        color_continuous_scale="RdYlGn",
        text=conf_acc["correct"].map(lambda x: f"{x:.0%}")
    )
    fig2.update_traces(textposition="outside")
    fig2.update_layout(
        height=420,
        margin=dict(t=60, b=110),
        xaxis_tickangle=-35,
        coloraxis_showscale=False
    )
    fig2.update_yaxes(range=[0, 1.05])
    st.plotly_chart(fig2, use_container_width=True)

st.markdown("---")

if class_metrics_df is not None and "label_code" in class_metrics_df.columns:
    st.subheader("Per-Class Performance")
    class_metrics = class_metrics_df[
        ~class_metrics_df["label_code"].isin(["accuracy", "macro avg", "weighted avg"])
    ].copy()
    class_metrics = class_metrics[pd.to_numeric(class_metrics["support"], errors="coerce") > 0]
    class_metrics = class_metrics.sort_values("f1-score", ascending=True)

    fig3 = px.bar(
        class_metrics,
        x="f1-score",
        y="label_code",
        orientation="h",
        color="f1-score",
        color_continuous_scale="RdYlGn",
        title="F1 Score per Label Class (Test Set)",
        labels={"f1-score": "F1 Score", "label_code": "Label Code"},
        height=max(900, len(class_metrics) * 22),
        text=class_metrics.apply(lambda r: f"{r['f1-score']:.2f} | n={int(r['support'])}", axis=1)
    )
    fig3.update_traces(textposition="outside")
    fig3.update_layout(
        yaxis={"categoryorder": "total ascending"},
        margin=dict(l=40, r=160, t=60, b=40),
        coloraxis_showscale=False,
    )
    st.plotly_chart(fig3, use_container_width=True)

    st.markdown("---")

    st.subheader("Precision vs Recall by Class")
    fig4 = px.scatter(
        class_metrics,
        x="precision",
        y="recall",
        size="support",
        color="f1-score",
        hover_name="label_code",
        color_continuous_scale="RdYlGn",
        title="Precision vs Recall (Bubble Size = Support)",
        labels={"precision": "Precision", "recall": "Recall"}
    )
    fig4.add_shape(type="line", x0=0, y0=0, x1=1, y1=1, line=dict(dash="dash", color="gray"))
    fig4.update_layout(height=520, margin=dict(t=60, b=40))
    st.plotly_chart(fig4, use_container_width=True)

    st.markdown("---")

st.subheader("Confusion Matrix (Top 20 Classes by Support)")
top20 = pred_df["true_label"].value_counts().head(20).index.tolist()
cm = confusion_matrix(pred_df["true_label"], pred_df["predicted_label"], labels=top20)
cm_top = pd.DataFrame(cm, index=top20, columns=top20)

fig5 = px.imshow(
    cm_top,
    color_continuous_scale="Blues",
    title="Confusion Matrix — Top 20 Classes",
    labels={"x": "Predicted", "y": "Actual", "color": "Count"},
    aspect="auto",
    height=720
)
fig5.update_xaxes(tickangle=-45)
fig5.update_layout(margin=dict(t=60, b=120, l=80, r=30))
st.plotly_chart(fig5, use_container_width=True)

st.markdown("---")

st.subheader("Sample Predictions with Confidence & Alternatives")
display_df = pred_df.copy().head(50)
for col in ["confidence", "alt_1_confidence", "alt_2_confidence"]:
    if col in display_df.columns:
        display_df[col] = display_df[col].map(lambda x: f"{x:.2%}" if pd.notnull(x) else "")

def highlight_correct(val):
    if val is True:
        return "background-color: rgba(46, 204, 113, 0.25)"
    if val is False:
        return "background-color: rgba(231, 76, 60, 0.25)"
    return ""

display_cols = [
    "signature", "predicted_label", "confidence",
    "alternative_1", "alt_1_confidence",
    "alternative_2", "alt_2_confidence",
    "true_label", "correct"
]
display_cols = [c for c in display_cols if c in display_df.columns]

st.dataframe(
    display_df[display_cols].style.applymap(
        highlight_correct,
        subset=["correct"] if "correct" in display_cols else []
    ),
    use_container_width=True,
    height=520
)

st.markdown("---")

st.subheader("Top Misclassified Label Pairs")
wrong = pred_df[pred_df["correct"] == False].copy()
if len(wrong) > 0:
    wrong["pair"] = wrong["true_label"] + " → " + wrong["predicted_label"]
    top_wrong = wrong["pair"].value_counts().head(15).reset_index()
    top_wrong.columns = ["Misclassification Pair", "Count"]

    fig6 = px.bar(
        top_wrong,
        x="Count",
        y="Misclassification Pair",
        orientation="h",
        color="Count",
        color_continuous_scale="Reds",
        title="Most Common Misclassifications (True → Predicted)",
        text="Count"
    )
    fig6.update_traces(textposition="outside")
    fig6.update_layout(
        yaxis={"categoryorder": "total ascending"},
        xaxis_range=[0, top_wrong["Count"].max() + 1],
        height=520,
        margin=dict(t=60, b=40, l=80, r=50),
        coloraxis_showscale=False,
    )
    st.plotly_chart(fig6, use_container_width=True)
else:
    st.info("No misclassifications found in predictions.csv")

st.markdown("---")

with st.expander("Model & Training Details", expanded=True):
    col1, col2 = st.columns(2)

    feature_shape = summary.get("feature_matrix_shape", {}) or {}
    text_features = summary.get("text_features", {}) or {}
    word_tfidf = text_features.get("word_tfidf", {}) or {}
    char_tfidf = text_features.get("char_tfidf", {}) or {}
    categorical_features = summary.get("categorical_features", {}) or {}
    numerical_features = summary.get("numerical_features", {}) or {}

    feature_shape_text = "—"
    if pd.notnull(feature_shape.get("rows")) and pd.notnull(feature_shape.get("cols")):
        feature_shape_text = f"({int(feature_shape['rows']):,}, {int(feature_shape['cols']):,})"

    projects_covered = summary.get("projects_covered")
    if isinstance(projects_covered, list):
        projects_covered_text = ", ".join(projects_covered) if projects_covered else "—"
    else:
        projects_covered_text = "—"

    mean_conf_text = f"{float(mean_confidence_val):.2%}" if pd.notnull(mean_confidence_val) else "—"

    with col1:
        st.markdown(f"""
        **Model:** {fmt_text_or_dash(summary.get("model_name", "LightGBM"))}  
        **Experiment:** {fmt_text_or_dash(summary.get("experiment"))}  
        **n_estimators:** {fmt_text_or_dash(summary.get("n_estimators"))}  
        **learning_rate:** {fmt_text_or_dash(summary.get("learning_rate"))}  
        **num_leaves:** {fmt_text_or_dash(summary.get("num_leaves"))}  
        **min_child_samples:** {fmt_text_or_dash(summary.get("min_child_samples"))}  
        **reg_alpha:** {fmt_text_or_dash(summary.get("reg_alpha"))}  
        **reg_lambda:** {fmt_text_or_dash(summary.get("reg_lambda"))}  
        **Class balancing:** `{fmt_text_or_dash(summary.get("class_weight"))}`  
        **Train signatures:** {fmt_int_or_dash(train_signatures_val)}  
        **Test signatures:** {fmt_int_or_dash(test_signatures_val)}  
        **Total label classes:** {fmt_int_or_dash(num_classes_val)}  
        **Accuracy:** {accuracy_val*100:.2f}%  
        **Macro F1:** {macro_f1_val:.4f}  
        **Weighted F1:** {weighted_f1_val:.4f}  
        **Mean confidence:** {mean_conf_text}  
        **Training time:** {fmt_float_or_dash(summary.get("train_time_s"), 1)} seconds  
        """)

    with col2:
        st.markdown(f"""
        **Text features:** {fmt_text_or_dash(text_features.get("method"))}  
        **Word TF-IDF:** max {fmt_text_or_dash(word_tfidf.get("max_features"))}, ngrams={fmt_text_or_dash(tuple(word_tfidf.get("ngram_range", [])) if word_tfidf.get("ngram_range") else None)}, sublinear_tf={fmt_text_or_dash(word_tfidf.get("sublinear_tf"))}, min_df={fmt_text_or_dash(word_tfidf.get("min_df"))}  
        **Char TF-IDF:** max {fmt_text_or_dash(char_tfidf.get("max_features"))}, analyzer={fmt_text_or_dash(char_tfidf.get("analyzer"))}, ngrams={fmt_text_or_dash(tuple(char_tfidf.get("ngram_range", [])) if char_tfidf.get("ngram_range") else None)}, sublinear_tf={fmt_text_or_dash(char_tfidf.get("sublinear_tf"))}, min_df={fmt_text_or_dash(char_tfidf.get("min_df"))}  
        **Categorical:** {fmt_text_or_dash(categorical_features.get("encoder"))} (`{fmt_text_or_dash(", ".join(categorical_features.get("columns", [])) if categorical_features.get("columns") else None)}`)  
        **Numerical:** {fmt_text_or_dash(numerical_features.get("scaler"))} + {fmt_text_or_dash(numerical_features.get("imputation"))} imputation (`{fmt_text_or_dash(", ".join(numerical_features.get("columns", [])) if numerical_features.get("columns") else None)}`)  
        **Split strategy:** {fmt_text_or_dash(summary.get("split_strategy"))}  
        **Leakage prevention:** {fmt_text_or_dash(summary.get("leakage_prevention"))}  
        **Feature matrix shape:** {feature_shape_text}  
        **Projects covered:** {projects_covered_text}  
        **Output source:** `bim_outputs/final`  
        """)
'''

with open("dashboard_updated.py", "w") as f:
    f.write(dashboard_streamlit_updated_code)

print("dashboard_updated.py written")
print("\nRun the dashboard with:")
print("  streamlit run dashboard_updated.py")


dashboard_updated.py written

Run the dashboard with:
  streamlit run dashboard_updated.py
